# 3.6 章节实践：从功能实现到流水优化的行 Softmax

## 概述

前面几节从 ReLU 的功能实现出发，逐步完成 tiling 分块、分核和多 buffer 流水，并通过 Add 对照了 SimdVF 与 SimtVF 的计算表达。本节继续沿用 SimdVF，以行 Softmax 作为本章的综合实践。

我们先把稳定 Softmax 拆成“求行最大值、减最大值、求指数、求行和、归一化”五步，完成单核功能和中间结果检查；再按完整行组成 tile，将多轮任务分给多个核；最后在相同分核实现上加入多 buffer 流水，并用 msprof 比较 kernel 耗时。三个层次使用同一数学公式，逐步把前面学过的知识串起来。

## 学习目标

- 理解稳定 Softmax 的五步计算，以及减去行最大值为什么不改变结果。
- 用 SimdVF 的归约与逐元素原语完成单核实现，并核对每一步的输出。
- 按完整行进行 tiling 和多核分配，保证每行的归约范围完整、数据覆盖不重不漏。
- 在相同分块分核基础上增加多 buffer 流水，结合 UB 用量和 msprof 数据评价优化。
- 将五步合并到一个 SimdVF，让中间结果保留在寄存器中，理解如何减少 UB 读写。

[本章导航](README.md) · [上一节](03.05_execution_modes.ipynb)

<p><small>运行前完成第 1.4 节的环境准备，并使用本章 README 指定的已验证运行入口启动新内核。每页独立执行，工作目录为本章根目录；内核本身不初始化 NPU。</small></p>

## 1. 实现思路分解：稳定 Softmax 的五步

输入 X 的形状为 `(M, C)`，M 是行数，C 是每行的元素数。行 Softmax 分别对每一行归一化：

$$
y_{ij}=\frac{e^{x_{ij}}}{\sum_{k=0}^{C-1}e^{x_{ik}}}
$$

直接计算大正数的指数可能溢出。令 $m_i=\max_k x_{ik}$，同一行的分子和分母同时乘以 $e^{-m_i}$，比例保持不变，得到稳定形式：

$$
y_{ij}=\frac{e^{x_{ij}-m_i}}{\sum_{k=0}^{C-1}e^{x_{ik}-m_i}}
$$

减最大值后，指数的输入都不大于 0。对本节有限 fp32 输入，至少一个指数项等于 1，行和为正；较小项可能下溢到 0，但不会出现直接对大正数求指数的溢出问题。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">步骤</th>
<th style="text-align: left; vertical-align: top;">数学计算</th>
<th style="text-align: left; vertical-align: top;">当前行 tile 内的形状</th>
<th style="text-align: left; vertical-align: top;">这一结果用于什么</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">1. 求最大值</td>
<td style="text-align: left; vertical-align: top;"><code>mx[i] = max_j a[i,j]</code></td>
<td style="text-align: left; vertical-align: top;"><code>(rows,)</code></td>
<td style="text-align: left; vertical-align: top;">每行得到一个平移基准</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">2. 减最大值</td>
<td style="text-align: left; vertical-align: top;"><code>shifted[i,j] = a[i,j] - mx[i]</code></td>
<td style="text-align: left; vertical-align: top;"><code>(rows,C)</code></td>
<td style="text-align: left; vertical-align: top;">把该行最大值平移到 0</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">3. 求指数</td>
<td style="text-align: left; vertical-align: top;"><code>e[i,j] = exp(shifted[i,j])</code></td>
<td style="text-align: left; vertical-align: top;"><code>(rows,C)</code></td>
<td style="text-align: left; vertical-align: top;">得到每项的非负权重</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">4. 求行和</td>
<td style="text-align: left; vertical-align: top;"><code>sm[i] = sum_j e[i,j]</code></td>
<td style="text-align: left; vertical-align: top;"><code>(rows,)</code></td>
<td style="text-align: left; vertical-align: top;">汇总该行指数权重</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">5. 归一化</td>
<td style="text-align: left; vertical-align: top;"><code>y[i,j] = e[i,j] / sm[i]</code></td>
<td style="text-align: left; vertical-align: top;"><code>(rows,C)</code></td>
<td style="text-align: left; vertical-align: top;">使该行输出之和接近 1</td>
</tr>
</tbody>
</table>

注意第四步求和的对象是 e，不是原始输入 a。`mx[i]` 和 `sm[i]` 都只属于第 i 行，广播时必须回到同一行。

![行归约与同一行的广播](images/03.06_softmax.png)

图中不同颜色代表不同的行：每行分别求最大值、做平移和指数，再对本行指数求和并归一化。mx 和 sm 各是一维 Tensor；shifted、e、y 保持二维形状。多核实现也必须维持这个行内对应关系，不能把一行的归约结果用于另一行。

### 用一行数据手算

用 `[1, 2, 3, 4]` 理解五步：最大值为 4，平移后是 `[-3, -2, -1, 0]`；指数约为 `[0.049787, 0.135335, 0.367879, 1]`；行和约为 `1.553002`；最后得到 `[0.032059, 0.087144, 0.236883, 0.643914]`，四项之和约为 1。这个四项例子只用于手算，后面的代码按完整的 64 列倍数处理输入。

## 2. 用到的原语与实践要求

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">已学知识或原语</th>
<th style="text-align: left; vertical-align: top;">在本节中的用途</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.Kernel</code>、tile 索引</td>
<td style="text-align: left; vertical-align: top;">启动逻辑 block，并确定每轮负责的行范围</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.alloc_shared</code>、<code>T.copy</code></td>
<td style="text-align: left; vertical-align: top;">分配 UB buffer，完成 GM 与 UB 的读入、写回</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.SimdVF</code>、<code>T.simd.vld/vsts</code></td>
<td style="text-align: left; vertical-align: top;">在 VF 内完成 UB 与寄存器之间的读写</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.simd.vsub/vexp/vdiv</code></td>
<td style="text-align: left; vertical-align: top;">在寄存器内计算减法、指数与除法</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.simd.vmax/vadd</code>、<code>T.simd.vcmax/vcadd</code></td>
<td style="text-align: left; vertical-align: top;">先合并多组寄存器，再归约寄存器中的各项</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>BRC_B32</code>、<code>ONEPT_B32</code></td>
<td style="text-align: left; vertical-align: top;">广播读入一个行标量、仅写回一个归约结果</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.serial</code></td>
<td style="text-align: left; vertical-align: top;">按顺序遍历当前 block 的多个行 tile</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.Pipelined</code>、<code>T.annotate_buffer_versions</code></td>
<td style="text-align: left; vertical-align: top;">安排多轮 tile 的流水，并保留多份 UB 存储</td>
</tr>
</tbody>
</table>

本节沿列维归约：输入 `(rows,C)`，输出 `(rows,)`。每行按 64 个 fp32 元素组成一组，`acc[0]` 是可更新的向量寄存器。

先用 `vmax` 或 `vadd` 合并各组同位置元素，再用 `vcmax` 或 `vcadd` 汇总寄存器内的各项；归约后，`one = T.simd.pset(32, "PAT_VL1")` 只启用一个 fp32 位置，配合 `ONEPT_B32` 写回一个行结果。

逐元素步骤用 `BRC_B32` 将同一行的最大值或行和广播到寄存器。五步仍分别放入 SimdVF 区域，由后端处理区域间的数据依赖。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">档位</th>
<th style="text-align: left; vertical-align: top;">输入示例与配置</th>
<th style="text-align: left; vertical-align: top;">必须完成的内容</th>
<th style="text-align: left; vertical-align: top;">相比上一档的变化</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">简单档</td>
<td style="text-align: left; vertical-align: top;"><code>(8,128)</code>，单核</td>
<td style="text-align: left; vertical-align: top;">完整五步 Softmax，逐个检查中间结果</td>
<td style="text-align: left; vertical-align: top;">先把算子功能实现正确</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">中等档</td>
<td style="text-align: left; vertical-align: top;"><code>(9216,128)</code>，rows=8，blocks=36</td>
<td style="text-align: left; vertical-align: top;">按完整行切 tile，多核覆盖全部输入</td>
<td style="text-align: left; vertical-align: top;">增加行索引、循环和分核</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">困难档</td>
<td style="text-align: left; vertical-align: top;">与中等档相同，stages=2/3</td>
<td style="text-align: left; vertical-align: top;">为同一个分核版本增加多 buffer 流水</td>
<td style="text-align: left; vertical-align: top;">改变流水与存储配置，保持数学计算不变</td>
</tr>
</tbody>
</table>

为了能逐步核对，下面三份教学实现统一写回五个输出：`Max、Shift、Exp、Sum、Y`。真正的 Softmax 结果是 Y，其他四份输出用于定位哪一步出现偏差。性能对照也保留相同的五个输出，不将它与只写回 Y 的实现直接混比。

### 输入与校验说明

本节示例代码处理连续、有限且绝对值不超过 2048 的 fp32 输入；这一范围与验证入口一致，用于限定本节已验证的输入域。

**当前教学入口要求 C 为 64 的倍数、rows 为 8 的倍数、M 能被 `rows × blocks` 整除。** 这些条件分别来自寄存器分组、行结果搬运的对齐安排和均匀分核的循环写法，逐项说明如下。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">输入条件</th>
<th style="text-align: left; vertical-align: top;">对应代码与来源</th>
<th style="text-align: left; vertical-align: top;">为什么在本例中这样选</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>C % 64 == 0</code></td>
<td style="text-align: left; vertical-align: top;"><code>T.serial(C // 64)</code>、<code>a[i, group * 64]</code> 以及全有效 <code>pset(32)</code></td>
<td style="text-align: left; vertical-align: top;">当前 256 字节向量寄存器一次容纳 64 个 fp32 元素。选完整组可直接读写并归约，省去尾列的掩码、越界访问保护和归约填充值处理。64 是列方向的寄存器粒度，不是行数或核数。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>rows % 8 == 0</code></td>
<td style="text-align: left; vertical-align: top;"><code>mx</code>、<code>sm</code> 的 shape 为 <code>(rows,)</code>，并通过 <code>T.copy</code> 写回 <code>Max</code>、<code>Sum</code></td>
<td style="text-align: left; vertical-align: top;">每行只有一个 fp32 归约结果，每份行向量的搬运长度是 <code>rows × 4</code> 字节。取 8 的倍数，使长度为 32 字节的倍数；配合整行 tile 的 <code>begin</code> 偏移，行结果写回的起点也按 32 字节对齐，便于沿用本例的对齐搬运路径。例如 rows=8 时，mx 和 sm 各写回 32 字节。这里的 8 来自行向量的搬运安排，不是要求归约同时处理 8 行，也不是 <code>PAT_VL1</code> 的限制。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>M % (rows * blocks) == 0</code></td>
<td style="text-align: left; vertical-align: top;">每核执行 <code>M // (rows * blocks)</code> 轮，<code>begin = bx * (M // blocks) + it * rows</code></td>
<td style="text-align: left; vertical-align: top;">本例把 M 行等分给 blocks 个逻辑 block，每个 block 每轮处理 rows 个完整行。整除保证每核行数为 rows 的倍数，既不漏掉尾行，也不需要不等长任务或边界分支。例如 <code>9216 / (8 × 36) = 32</code>，每核恰好执行 32 轮。单核版本取 rows=M、blocks=1，此条件自然成立。</td>
</tr>
</tbody>
</table>

这些倍数与整除约定**只是为了便于示例讲解与校验，并非 Softmax 算法或 TileLang 的固有限制**。支持更一般的形状时，可以为尾列增加有效元素掩码与安全搬运，必要时为最大值归约填充负无穷、为求和归约填充零；为行结果采用补齐存储或合适的非对齐搬运；为尾行增加边界判断并调整各核任务分配。具体仍需满足所选指令与后端的要求并重新验证。当前入口尚未实现这些分支，所以不满足上述条件时仍会校验拦截。

fp32 类型和连续布局对应示例中固定的 Tensor 声明与连续地址访问；有限值及绝对值不超过 2048 是本课程验证入口选定的输入域，不是 Softmax 公式的幅值上限。此外，M、C、rows、blocks、versions 都必须为正，分别保证形状、每轮行数、逻辑 block 数及 buffer 份数有效。

每份 UB 存储包括 a、shifted、e、y 四个矩阵，以及 mx、sm 两个行向量。按元素容量计算：`UB 字节数 = 4 × (4 × rows × C + 2 × rows) × versions`。当前环境可分配的UB容量为 248 KiB，下面的入口据此检查这些 buffer 的元素容量；增加 buffer 份数也必须重新核算。容量约束仍需遵守，不能因为前面的形状约定是教学简化就忽略它。统一采用下面的校验。

```python
import tilelang
import tilelang.language as T


def check_softmax_config(M, C, rows, blocks, versions=1):
    if min(M, C, rows, blocks, versions) <= 0 or C % 64 or rows % 8 or M % (rows * blocks):
        raise ValueError("Require C % 64 == 0, rows % 8 == 0, M % (rows * blocks) == 0")
    required = 4 * (4 * rows * C + 2 * rows) * versions
    if required > 248 * 1024:
        raise ValueError("Six fp32 buffers exceed the 248 KiB SimdVF UB budget")
```

### 统一的精度检查

随机值、全零、常数和每行覆盖 −1000 到 1000 的大幅值输入分别检查五个完整输出。行最大值和减法使用精确比较；指数、求和、归一化使用 `rtol=1e-5, atol=1e-6`。最后还要与 `torch.softmax` 对照，检查输出有限、非负且每行和接近 1。

**正确性验证**：单核、串行多核与双/三 buffer 版本均已完成四类输入验证；五个完整输出、torch.softmax 对照、非负有限及行和检查全部通过。

## 3. 简单档：单核功能实现

先处理 `(8,128)` 的完整输入，只启动一个逻辑 block，不引入分核和多轮 tile。读入 A 后依次完成五步，再将五个结果写回。沿用上一小节的 import 与公共检查，完整实现如下。

```python
def softmax_single(M=8, C=128):
    rows, blocks = M, 1
    check_softmax_config(M, C, rows, blocks)

    @T.prim_func
    def main(
        A: T.Tensor((M, C), "float32"),
        Max: T.Tensor((M,), "float32"),
        Shift: T.Tensor((M, C), "float32"),
        Exp: T.Tensor((M, C), "float32"),
        Sum: T.Tensor((M,), "float32"),
        Y: T.Tensor((M, C), "float32"),
    ):
        with T.Kernel(blocks) as bx:
            a = T.alloc_shared((rows, C), 'float32')
            mx = T.alloc_shared((rows,), 'float32')
            shifted = T.alloc_shared((rows, C), 'float32')
            e = T.alloc_shared((rows, C), 'float32')
            sm = T.alloc_shared((rows,), 'float32')
            y = T.alloc_shared((rows, C), 'float32')
            T.copy(A, a)

            with T.SimdVF():
                mask = T.simd.pset(32)
                one = T.simd.pset(32, "PAT_VL1")
                acc = T.simd.alloc_local((1,), "float32")
                for i in T.serial(rows):
                    acc[0] = T.simd.vld(a[i, 0])
                    for group in T.serial(1, C // 64):
                        ra = T.simd.vld(a[i, group * 64])
                        acc[0] = T.simd.vmax(acc[0], ra, mask)
                    row_max = T.simd.vcmax(acc[0], mask)
                    T.simd.vsts(mx[i], row_max, one, "ONEPT_B32")

            with T.SimdVF():
                mask = T.simd.pset(32)
                for i in T.serial(rows):
                    rmax = T.simd.vld(mx[i], "BRC_B32")
                    for group in T.serial(C // 64):
                        ra = T.simd.vld(a[i, group * 64])
                        rs = T.simd.vsub(ra, rmax, mask)
                        T.simd.vsts(shifted[i, group * 64], rs, mask)

            with T.SimdVF():
                mask = T.simd.pset(32)
                for i in T.serial(rows):
                    for group in T.serial(C // 64):
                        rs = T.simd.vld(shifted[i, group * 64])
                        re = T.simd.vexp(rs, mask)
                        T.simd.vsts(e[i, group * 64], re, mask)

            with T.SimdVF():
                mask = T.simd.pset(32)
                one = T.simd.pset(32, "PAT_VL1")
                acc = T.simd.alloc_local((1,), "float32")
                for i in T.serial(rows):
                    acc[0] = T.simd.vld(e[i, 0])
                    for group in T.serial(1, C // 64):
                        re = T.simd.vld(e[i, group * 64])
                        acc[0] = T.simd.vadd(acc[0], re, mask)
                    row_sum = T.simd.vcadd(acc[0], mask)
                    T.simd.vsts(sm[i], row_sum, one, "ONEPT_B32")

            with T.SimdVF():
                mask = T.simd.pset(32)
                for i in T.serial(rows):
                    rsum = T.simd.vld(sm[i], "BRC_B32")
                    for group in T.serial(C // 64):
                        re = T.simd.vld(e[i, group * 64])
                        ry = T.simd.vdiv(re, rsum, mask)
                        T.simd.vsts(y[i, group * 64], ry, mask)

            T.copy(mx, Max)
            T.copy(shifted, Shift)
            T.copy(e, Exp)
            T.copy(sm, Sum)
            T.copy(y, Y)

    return tilelang.compile(main, out_idx=[1, 2, 3, 4, 5])
```

### 分步阅读代码

先看准备与读入：`rows=M=8`，a、shifted、e、y 的 shape 为 `(8,128)`，mx、sm 的 shape 为 `(8,)`。`T.copy(A,a)` 把八行输入从 GM 拷入 UB。接下来的五个 `T.SimdVF()` 按顺序对应五个数学步骤；每个区域中的 `T.serial(rows)` 都逐行处理，每行的归约互相独立。

#### 第 1 个 SimdVF：求每行最大值，a → mx

`mask = T.simd.pset(32)` 生成按 32 位元素解释的全有效掩码，覆盖一个寄存器中的 64 个 fp32 位置。`acc = T.simd.alloc_local((1,), "float32")` 分配一个可更新的向量寄存器槽位；这里 `(1,)` 表示一个寄存器，`acc[0]` 仍包含 64 个 fp32 元素，并不是一个 fp32 标量。

每行先用 `vld(a[i, 0])` 把前 64 项读入 acc，再从第 1 组开始读取其余组，避免重复合并第一组。`vmax(acc[0], ra, mask)` 对两个寄存器的**对应位置逐项取最大值**，输出仍是一组 64 项；它本身不会得到整行最大值。随后 `vcmax(acc[0], mask)` 对这一组中所有有效位置做**横向最大值归约**，把归约结果放在返回寄存器的第 0 个位置。

例如 C=128 时，`vmax` 得到 `acc[k] = max(a[i,k], a[i,k+64])`（k=0～63）；再用 `vcmax` 对这 64 个候选值取最大值，才覆盖整行 128 项。C=64 时无需组间合并，但仍需要 `vcmax` 完成组内归约。

最后的 `vsts(mx[i], row_max, one, "ONEPT_B32")` 将本行结果保存到 UB 的 mx[i]，其中两个参数分工不同：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">参数</th>
<th style="text-align: left; vertical-align: top;">控制内容</th>
<th style="text-align: left; vertical-align: top;">在这里的作用</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>one = pset(32, "PAT_VL1")</code></td>
<td style="text-align: left; vertical-align: top;">谓词掩码：哪些元素位置有效</td>
<td style="text-align: left; vertical-align: top;">按 32 位元素粒度只启用第 0 个位置。它不做归约、不选择地址，也不把标量广播成整组。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>"ONEPT_B32"</code></td>
<td style="text-align: left; vertical-align: top;">存储分布模式：怎样把寄存器内容写到内存</td>
<td style="text-align: left; vertical-align: top;">采用单个 32 位元素的写回方式，把归约结果存入 mx[i]，而不是按默认 <code>NORM_B32</code> 写出一组连续 fp32 元素。</td>
</tr>
</tbody>
</table>

因此，PAT_VL1 表达“只启用一个位置”，ONEPT_B32 表达“按单个 32 位元素存储”；二者配合，让每行只占 mx 中的一个元素，不覆盖相邻行的结果。归约时仍用全有效 mask，不能提前换成 one，否则只会归约一个位置。

#### 第 2 个 SimdVF：减去本行最大值，a、mx → shifted

`vld(mx[i], "BRC_B32")` 从 UB 读取本行的一个 32 位最大值，并广播到寄存器的全部 64 个位置；它不会连续读取 mx 中 64 行的最大值。随后逐组 `vld` 读取 a，`vsub(ra, rmax, mask)` 逐项计算 `a[i,j] - mx[i]`，`vsts` 按默认连续存储方式把结果写入 shifted。每一组都使用同一行的最大值，所以该行的最大项被平移到 0。

#### 第 3 个 SimdVF：逐项求指数，shifted → e

每行每组用 `vld` 读入 shifted，用 `vexp(rs, mask)` 对有效位置分别计算指数，再用 `vsts` 写到 e。这里没有归约，也不改变矩阵形状；e 的每个元素对应 `exp(a[i,j] - mx[i])`，供后面求和与归一化共同使用。

#### 第 4 个 SimdVF：求本行指数和，e → sm

与第 1 个区域的结构对应，先将 e 的第一组读入 acc，再用 `vadd(acc[0], re, mask)` 将其余组的对应位置逐项相加。`vadd` 输出仍有 64 项；`vcadd(acc[0], mask)` 才把这些有效位置横向求和，将整行指数和放在结果寄存器的第 0 个位置。例如 C=128 时，先得到 `acc[k] = e[i,k] + e[i,k+64]`，再把 k=0～63 求和。

最后用 PAT_VL1 掩码与 ONEPT_B32 单元素存储，将结果写入 sm[i]。这里归约的是 e，不能误用 a 或 shifted；浮点求和顺序也可能与参考实现不同，因此按前文的固定容差检查。

#### 第 5 个 SimdVF：除以本行指数和，e、sm → y

`vld(sm[i], "BRC_B32")` 将本行的一个行和广播到 64 个位置。每组读入 e 后，`vdiv(re, rsum, mask)` 逐项计算 `e[i,j] / sm[i]`，再用 `vsts` 写入 y。对本节有限输入，减最大值后至少有一个指数项为 1，所以本行分母为正。

最后回到 GM 写回：五次 `T.copy` 分别输出 Max、Shift、Exp、Sum、Y；`out_idx=[1,2,3,4,5]` 让调用返回这五份 Tensor。五个 VF 通过 UB 中间结果衔接，区域之间的数据依赖由后端处理；vld/vsts 负责 UB 与寄存器之间的读写，T.copy 负责这里的 GM 与 UB 之间的搬运。

本配置六份 buffer 的元素容量合计为 16448 B，约 16.0625 KiB。调用 `softmax_single(8,128)` 得到可执行 kernel，再按四类输入检查完整结果。完整答案入口为 [03.06_easy.py](answer/03.06_easy.py)。

In [1]:
import subprocess, sys, re
from pathlib import Path
from src.course_paths import OUTPUT_DIR
(OUTPUT_DIR / "validation").mkdir(parents=True, exist_ok=True)

def run_checked(script):
    try:
        result = subprocess.run([sys.executable, script], check=True,
                                capture_output=True, text=True)
    except subprocess.CalledProcessError as error:
        print(error.stdout)
        print(error.stderr)
        raise
    Path(OUTPUT_DIR / "validation", Path(script).stem + "_notebook.log").write_text(
        result.stdout + result.stderr)
    print("\n".join(line for line in result.stdout.splitlines()
                    if not re.match(r"^\d{4}-\d{2}-\d{2}", line)
                    and not line.startswith("Loading tilelang libs from dev root:")))

run_checked("answer/03.06_easy.py")

easy single random amax: full-output PASS; max_abs_error=0
easy single random subtract: full-output PASS; max_abs_error=0
easy single random exp: full-output PASS; max_abs_error=5.9604645e-08
easy single random sum: full-output PASS; max_abs_error=1.9073486e-06
easy single random divide: full-output PASS; max_abs_error=7.4505806e-09
easy single random: torch.softmax, finite/nonnegative output and row-sum checks PASS
easy single zero amax: full-output PASS; max_abs_error=0
easy single zero subtract: full-output PASS; max_abs_error=0
easy single zero exp: full-output PASS; max_abs_error=0
easy single zero sum: full-output PASS; max_abs_error=0
easy single zero divide: full-output PASS; max_abs_error=0
easy single zero: torch.softmax, finite/nonnegative output and row-sum checks PASS
easy single constant amax: full-output PASS; max_abs_error=0
easy single constant subtract: full-output PASS; max_abs_error=0
easy single constant exp: full-output PASS; max_abs_error=0
easy single constant s

## 4. 中等档：tiling 多核切分

把输入扩大到 `(9216,128)`，若仍把全部行放入单份 UB，按相同公式需要约 18.07 MiB。因此像 3.3 一样控制单次处理的数据量，但这里要以**完整行**为单位分块。

每个行 tile 包含 8 行和全部 128 列，单份 UB 用量仍约 16.0625 KiB。不要把同一行的不同列直接交给不同核、各自求局部 Softmax 后拼接，因为每一项都依赖该行完整的最大值与指数和。

沿用 3.5 扩展用例的 36 个向量核任务作为本节配置；该数值是实验选择，不是设备的向量核总数。先将 9216 行均分给 36 个逻辑 block，每核负责 256 个完整行；再在核内每次处理 8 行，共 32 轮。行首索引仍沿用熟悉的形式：`begin = bx * (M // blocks) + it * rows`。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">it</th>
<th style="text-align: left; vertical-align: top;">bx</th>
<th style="text-align: left; vertical-align: top;">处理的行区间，左闭右开</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">[0, 8)</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">[256, 264)</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">35</td>
<td style="text-align: left; vertical-align: top;">[8960, 8968)</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;">[8, 16)</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">31</td>
<td style="text-align: left; vertical-align: top;">35</td>
<td style="text-align: left; vertical-align: top;">[9208, 9216)</td>
</tr>
</tbody>
</table>

相同 it 的 block 相互独立；同一 block 按 it 遍历自己负责的行 tile。每个 tile 保留整行的列范围，所以五步计算不需要跨核合并归约结果。

### 本档新增的索引与循环

下面只摘出行索引、读入和最终结果写回；四份中间结果也使用同一行偏移。

```python
for it in T.serial(M // (rows * blocks)):
    begin = bx * (M // blocks) + it * rows
    T.copy(A[begin : begin + rows, 0:C], a)
    # 此处仍执行单核版本的五步计算。
    T.copy(y, Y[begin : begin + rows, 0:C])
```

<details>
<summary>展开中等档完整实现（五步计算与单核版本相同）</summary>

```python
def softmax_kernel(M, C, rows=8, blocks=1):
    check_softmax_config(M, C, rows, blocks)

    @T.prim_func
    def main(
        A: T.Tensor((M, C), "float32"),
        Max: T.Tensor((M,), "float32"),
        Shift: T.Tensor((M, C), "float32"),
        Exp: T.Tensor((M, C), "float32"),
        Sum: T.Tensor((M,), "float32"),
        Y: T.Tensor((M, C), "float32"),
    ):
        with T.Kernel(blocks) as bx:
            a = T.alloc_shared((rows, C), 'float32')
            mx = T.alloc_shared((rows,), 'float32')
            shifted = T.alloc_shared((rows, C), 'float32')
            e = T.alloc_shared((rows, C), 'float32')
            sm = T.alloc_shared((rows,), 'float32')
            y = T.alloc_shared((rows, C), 'float32')

            for it in T.serial(M // (rows * blocks)):
                begin = bx * (M // blocks) + it * rows
                T.copy(A[begin : begin + rows, 0:C], a)

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    one = T.simd.pset(32, "PAT_VL1")
                    acc = T.simd.alloc_local((1,), "float32")
                    for i in T.serial(rows):
                        acc[0] = T.simd.vld(a[i, 0])
                        for group in T.serial(1, C // 64):
                            ra = T.simd.vld(a[i, group * 64])
                            acc[0] = T.simd.vmax(acc[0], ra, mask)
                        row_max = T.simd.vcmax(acc[0], mask)
                        T.simd.vsts(mx[i], row_max, one, "ONEPT_B32")

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    for i in T.serial(rows):
                        rmax = T.simd.vld(mx[i], "BRC_B32")
                        for group in T.serial(C // 64):
                            ra = T.simd.vld(a[i, group * 64])
                            rs = T.simd.vsub(ra, rmax, mask)
                            T.simd.vsts(shifted[i, group * 64], rs, mask)

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    for i in T.serial(rows):
                        for group in T.serial(C // 64):
                            rs = T.simd.vld(shifted[i, group * 64])
                            re = T.simd.vexp(rs, mask)
                            T.simd.vsts(e[i, group * 64], re, mask)

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    one = T.simd.pset(32, "PAT_VL1")
                    acc = T.simd.alloc_local((1,), "float32")
                    for i in T.serial(rows):
                        acc[0] = T.simd.vld(e[i, 0])
                        for group in T.serial(1, C // 64):
                            re = T.simd.vld(e[i, group * 64])
                            acc[0] = T.simd.vadd(acc[0], re, mask)
                        row_sum = T.simd.vcadd(acc[0], mask)
                        T.simd.vsts(sm[i], row_sum, one, "ONEPT_B32")

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    for i in T.serial(rows):
                        rsum = T.simd.vld(sm[i], "BRC_B32")
                        for group in T.serial(C // 64):
                            re = T.simd.vld(e[i, group * 64])
                            ry = T.simd.vdiv(re, rsum, mask)
                            T.simd.vsts(y[i, group * 64], ry, mask)

                T.copy(mx, Max[begin : begin + rows])
                T.copy(shifted, Shift[begin : begin + rows, 0:C])
                T.copy(e, Exp[begin : begin + rows, 0:C])
                T.copy(sm, Sum[begin : begin + rows])
                T.copy(y, Y[begin : begin + rows, 0:C])

    return tilelang.compile(main, out_idx=[1, 2, 3, 4, 5])
```

</details>

### 对照单核版本理解变化

UB 的 rows 固定为 8；GM Tensor 的 M 可以更大。读入和五次写回都加上同一个 `begin` 行偏移，tile 内的归约维度与五步公式完全保留。外层 `T.serial` 遍历多轮，内部 `T.serial` 遍历行和寄存器组，寄存器微指令完成逐元素计算及行归约。

先用 `softmax_kernel(9216,128,8,1)` 核对覆盖范围，再用 `softmax_kernel(9216,128,8,36)` 检查多核结果。两者每行的计算一致，区别是每个 block 分到的轮数。完整答案入口为 [03.06_medium.py](answer/03.06_medium.py)。

In [2]:
run_checked("answer/03.06_medium.py")

medium serial blocks=1 random amax: full-output PASS; max_abs_error=0
medium serial blocks=1 random subtract: full-output PASS; max_abs_error=0
medium serial blocks=1 random exp: full-output PASS; max_abs_error=5.9604645e-08
medium serial blocks=1 random sum: full-output PASS; max_abs_error=5.7220459e-06
medium serial blocks=1 random divide: full-output PASS; max_abs_error=5.9604645e-08
medium serial blocks=1 random: torch.softmax, finite/nonnegative output and row-sum checks PASS
medium serial blocks=1 zero amax: full-output PASS; max_abs_error=0
medium serial blocks=1 zero subtract: full-output PASS; max_abs_error=0
medium serial blocks=1 zero exp: full-output PASS; max_abs_error=0
medium serial blocks=1 zero sum: full-output PASS; max_abs_error=0
medium serial blocks=1 zero divide: full-output PASS; max_abs_error=0
medium serial blocks=1 zero: torch.softmax, finite/nonnegative output and row-sum checks PASS
medium serial blocks=1 constant amax: full-output PASS; max_abs_error=0
medi

## 5. 困难档：流水排布

中等档每个 block 已有 32 轮 tile，可以应用 3.4 的多 buffer 思路：让不同轮次使用不同的 UB 内存，为拷入、计算与拷出的交叠创造条件。五步之间的数据依赖仍要保留；流水优化不会让同一行的求和早于指数完成。

这里固定 `M=9216, C=128, rows=8, blocks=36`，对六个 UB buffer 都配置内存份数，将 `T.serial` 改为 `T.Pipelined`。输入 a 和结果 y 需要避免过早覆盖，mx、shifted、e、sm 也有跨步骤使用和教学写回，所以同样配置对应的内存份数。

### 本档新增的流水配置

下面只摘出 buffer 配置与循环入口，计算主体保持不变。

```python
T.annotate_buffer_versions({a: stages, mx: stages, shifted: stages, e: stages, sm: stages, y: stages})

for it in T.Pipelined(M // (rows * blocks), num_stages=stages):
    begin = bx * (M // blocks) + it * rows
    # 读入、五步计算与五份结果写回均沿用中等档。
```

<details>
<summary>展开困难档完整实现（保留五步计算与五份输出）</summary>

```python
def softmax_pipeline(M, C, rows=8, blocks=36, stages=2):
    check_softmax_config(M, C, rows, blocks, stages)

    @T.prim_func
    def main(
        A: T.Tensor((M, C), "float32"),
        Max: T.Tensor((M,), "float32"),
        Shift: T.Tensor((M, C), "float32"),
        Exp: T.Tensor((M, C), "float32"),
        Sum: T.Tensor((M,), "float32"),
        Y: T.Tensor((M, C), "float32"),
    ):
        with T.Kernel(blocks) as bx:
            a = T.alloc_shared((rows, C), 'float32')
            mx = T.alloc_shared((rows,), 'float32')
            shifted = T.alloc_shared((rows, C), 'float32')
            e = T.alloc_shared((rows, C), 'float32')
            sm = T.alloc_shared((rows,), 'float32')
            y = T.alloc_shared((rows, C), 'float32')
            T.annotate_buffer_versions({a: stages, mx: stages, shifted: stages, e: stages, sm: stages, y: stages})

            for it in T.Pipelined(M // (rows * blocks), num_stages=stages):
                begin = bx * (M // blocks) + it * rows
                T.copy(A[begin : begin + rows, 0:C], a)

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    one = T.simd.pset(32, "PAT_VL1")
                    acc = T.simd.alloc_local((1,), "float32")
                    for i in T.serial(rows):
                        acc[0] = T.simd.vld(a[i, 0])
                        for group in T.serial(1, C // 64):
                            ra = T.simd.vld(a[i, group * 64])
                            acc[0] = T.simd.vmax(acc[0], ra, mask)
                        row_max = T.simd.vcmax(acc[0], mask)
                        T.simd.vsts(mx[i], row_max, one, "ONEPT_B32")

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    for i in T.serial(rows):
                        rmax = T.simd.vld(mx[i], "BRC_B32")
                        for group in T.serial(C // 64):
                            ra = T.simd.vld(a[i, group * 64])
                            rs = T.simd.vsub(ra, rmax, mask)
                            T.simd.vsts(shifted[i, group * 64], rs, mask)

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    for i in T.serial(rows):
                        for group in T.serial(C // 64):
                            rs = T.simd.vld(shifted[i, group * 64])
                            re = T.simd.vexp(rs, mask)
                            T.simd.vsts(e[i, group * 64], re, mask)

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    one = T.simd.pset(32, "PAT_VL1")
                    acc = T.simd.alloc_local((1,), "float32")
                    for i in T.serial(rows):
                        acc[0] = T.simd.vld(e[i, 0])
                        for group in T.serial(1, C // 64):
                            re = T.simd.vld(e[i, group * 64])
                            acc[0] = T.simd.vadd(acc[0], re, mask)
                        row_sum = T.simd.vcadd(acc[0], mask)
                        T.simd.vsts(sm[i], row_sum, one, "ONEPT_B32")

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    for i in T.serial(rows):
                        rsum = T.simd.vld(sm[i], "BRC_B32")
                        for group in T.serial(C // 64):
                            re = T.simd.vld(e[i, group * 64])
                            ry = T.simd.vdiv(re, rsum, mask)
                            T.simd.vsts(y[i, group * 64], ry, mask)

                T.copy(mx, Max[begin : begin + rows])
                T.copy(shifted, Shift[begin : begin + rows, 0:C])
                T.copy(e, Exp[begin : begin + rows, 0:C])
                T.copy(sm, Sum[begin : begin + rows])
                T.copy(y, Y[begin : begin + rows, 0:C])

    return tilelang.compile(main, out_idx=[1, 2, 3, 4, 5])
```

</details>

### 分步解析与 UB 检查

- **分核不变**：`begin` 仍由 it、bx 和 rows 决定，每个 block 仍然处理 32 轮完整行 tile。
- **计算不变**：五个 SimdVF 区域、归约维度和五份输出保持一致。
- **存储改变**：双 buffer 让第 0、2、4 轮使用一组 UB 内存，第 1、3、5 轮使用另一组；具体复用依赖由编译器处理。
- **流水改变**：`T.Pipelined` 给出多轮迭代的流水配置。增加 buffer 后仍要先确认编译和完整输出正确，再采集耗时。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">配置</th>
<th style="text-align: left; vertical-align: top;">六份 buffer 的元素容量</th>
<th style="text-align: left; vertical-align: top;">正确性验证状态</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">中等档，单 buffer</td>
<td style="text-align: left; vertical-align: top;">16.0625 KiB</td>
<td style="text-align: left; vertical-align: top;">四类输入全部通过</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">困难档，双 buffer</td>
<td style="text-align: left; vertical-align: top;">32.125 KiB</td>
<td style="text-align: left; vertical-align: top;">四类输入全部通过</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">困难档，三 buffer</td>
<td style="text-align: left; vertical-align: top;">48.1875 KiB</td>
<td style="text-align: left; vertical-align: top;">四类输入全部通过</td>
</tr>
</tbody>
</table>

这些配置的元素容量都低于当前环境可分配的UB容量（248 KiB）；实际分配与流水组合已完成编译和 NPU 验证。完整答案入口为 [03.06_hard.py](answer/03.06_hard.py)，其中分别检查双、三 buffer 的五个完整输出。

In [3]:
run_checked("answer/03.06_hard.py")

hard pipeline stages=2 random amax: full-output PASS; max_abs_error=0
hard pipeline stages=2 random subtract: full-output PASS; max_abs_error=0
hard pipeline stages=2 random exp: full-output PASS; max_abs_error=5.9604645e-08
hard pipeline stages=2 random sum: full-output PASS; max_abs_error=5.7220459e-06
hard pipeline stages=2 random divide: full-output PASS; max_abs_error=5.9604645e-08
hard pipeline stages=2 random: torch.softmax, finite/nonnegative output and row-sum checks PASS
hard pipeline stages=2 zero amax: full-output PASS; max_abs_error=0
hard pipeline stages=2 zero subtract: full-output PASS; max_abs_error=0
hard pipeline stages=2 zero exp: full-output PASS; max_abs_error=0
hard pipeline stages=2 zero sum: full-output PASS; max_abs_error=0
hard pipeline stages=2 zero divide: full-output PASS; max_abs_error=0
hard pipeline stages=2 zero: torch.softmax, finite/nonnegative output and row-sum checks PASS
hard pipeline stages=2 constant amax: full-output PASS; max_abs_error=0
hard

### 用相同工作量评价流水

沿用 3.4 的 msprof 口径，比较中等档串行版本与困难档双、三 buffer 版本的 `Task Duration(us)`。三者的 shape、tile、block 数、输入与五份输出均相同，不再用“rows 从 8 改 16”的 Host 调用耗时作为困难档成绩。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">实现</th>
<th style="text-align: left; vertical-align: top;">rows / blocks</th>
<th style="text-align: left; vertical-align: top;">buffer 份数</th>
<th style="text-align: left; vertical-align: top;">kernel 中位数 μs</th>
<th style="text-align: left; vertical-align: top;">相对中等档加速比</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">中等档 <code>T.serial</code></td>
<td style="text-align: left; vertical-align: top;">8 / 36</td>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">29.7310</td>
<td style="text-align: left; vertical-align: top;">1.00</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">困难档 <code>T.Pipelined</code></td>
<td style="text-align: left; vertical-align: top;">8 / 36</td>
<td style="text-align: left; vertical-align: top;">2</td>
<td style="text-align: left; vertical-align: top;">10.9660</td>
<td style="text-align: left; vertical-align: top;">2.71</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">困难档 <code>T.Pipelined</code></td>
<td style="text-align: left; vertical-align: top;">8 / 36</td>
<td style="text-align: left; vertical-align: top;">3</td>
<td style="text-align: left; vertical-align: top;">11.5635</td>
<td style="text-align: left; vertical-align: top;">2.57</td>
</tr>
</tbody>
</table>

```bash
source src/env.sh
for mode in serial 2 3; do
  msprof --output="${COURSE_OUTPUT_DIR}/profiles/msprof_03_06/${mode}" \
    --task-time=on --ai-core=off --runtime-api=on \
    python src/profile_softmax.py "$mode"
done
```

每配置先校验一次，再做三批“10 次预热 + 30 次正式采样”，汇总 90 个目标 kernel 样本。每批结束后检查五份输出；按实际导出的目标 kernel 记录筛选，不计 Host 侧开销。最终用“中等档中位数 / 当前配置中位数”计算加速比，并结合 UB 用量解释结果；没有收益时也应如实记录。



表中保留课程参考性能数据；在当前环境复测时，以新生成的 msprof 记录为准。完整源文件为 [softmax_kernel.py](src/softmax_kernel.py)，统一验证见 [run_softmax.py](src/run_softmax.py)，采集入口见 [profile_softmax.py](src/profile_softmax.py)。
此前寄存器微指令实现的测量中，双 buffer 的加速比为 2.71，三 buffer 为 2.57。本例保留五份教学输出，结论对应这一完整工作量。增加 buffer 并不保证继续提速，应按相同工作量下的实测结果选择。

## 6. 五步合一：在一个 SimdVF 中完成 Softmax

前面把 Softmax 拆成五个执行区，是为了逐步观察行最大值、平移值、指数值、行和与最终结果。**五个数学步骤不需要对应五个 SimdVF。** 对于本章每行 128 个 fp32 元素的用例，可以逐行将数据放入两个寄存器组，在同一个 SimdVF 中连续完成五步计算。

关键不只是把五段代码放进同一对 `with T.SimdVF()` 中，还要让中间结果直接留在寄存器中供后续指令使用。精简版不再分配 mx、shifted、e、sm 四个 UB buffer，也不再把这些中间结果写回 GM；只保留输入 a 和输出 y，最终返回 Y。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">数学步骤</th>
<th style="text-align: left; vertical-align: top;">精简版中的实现</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">求行最大值</td>
<td style="text-align: left; vertical-align: top;">先用 <code>vmax(x0, x1)</code> 合并两组对应位置，再用 <code>vcmax</code> 归约整组，得到整行最大值</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">减最大值</td>
<td style="text-align: left; vertical-align: top;"><code>vdupv</code> 将归约结果的第 0 个 lane 广播到整组，随后用 <code>vsub</code> 同时处理两半行</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">求指数</td>
<td style="text-align: left; vertical-align: top;"><code>vexp</code> 直接接收减法结果，e0、e1 留在寄存器中</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">求行和</td>
<td style="text-align: left; vertical-align: top;">先用 <code>vadd(e0, e1)</code> 合并两半行，再用 <code>vcadd</code> 求出 128 项的总和</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">归一化</td>
<td style="text-align: left; vertical-align: top;">用 <code>vdupv</code> 广播行和，执行 <code>vdiv</code>，仅将最终结果存入 y</td>
</tr>
</tbody>
</table>

`vdupv` 在寄存器之间完成广播，不需要先把行最大值或行和写到 UB，再通过 `BRC_B32` 读回来。这里两组对应位置先合并、再做组内归约，覆盖的是整行 128 项；不能分别对两个 64 项片段计算 Softmax 后直接拼接。

### 五步合一的精简实现

下面保留前面的分核、按行分块和多 buffer 配置，每个 tile 只使用一个 SimdVF。默认处理 `(9216,128)`，每轮 8 行、36 个核、双 buffer；每行依次处理，寄存器空间可在不同行之间复用。本例固定列数为 128，不处理尾块。

```python
import tilelang
import tilelang.language as T


def softmax_fused(M=9216, rows=8, blocks=36, stages=2):
    if min(M, rows, blocks, stages) <= 0 or rows % 8 or M % (rows * blocks):
        raise ValueError("Require rows % 8 == 0 and M % (rows * blocks) == 0")
    if 2 * rows * 128 * 4 * stages > 248 * 1024:
        raise ValueError("Input/output buffers exceed the 248 KiB UB budget")

    @T.prim_func
    def main(
        A: T.Tensor((M, 128), "float32"),
        Y: T.Tensor((M, 128), "float32"),
    ):
        with T.Kernel(blocks) as bx:
            a = T.alloc_shared((rows, 128), "float32")
            y = T.alloc_shared((rows, 128), "float32")
            T.annotate_buffer_versions({a: stages, y: stages})

            for it in T.Pipelined(M // (rows * blocks), num_stages=stages):
                begin = bx * (M // blocks) + it * rows
                T.copy(A[begin : begin + rows, :], a)

                with T.SimdVF():
                    mask = T.simd.pset(32)
                    for i in T.serial(rows):
                        x0 = T.simd.vld(a[i, 0])
                        x1 = T.simd.vld(a[i, 64])
                        # 1. Row maximum; broadcast lane 0 to every lane.
                        mx = T.simd.vcmax(T.simd.vmax(x0, x1, mask), mask)
                        mx_all = T.simd.vdupv(mx, mask)
                        # 2. Subtract maximum. 3. Exponentiate.
                        e0 = T.simd.vexp(T.simd.vsub(x0, mx_all, mask), mask)
                        e1 = T.simd.vexp(T.simd.vsub(x1, mx_all, mask), mask)
                        # 4. Row sum; broadcast it within the registers.
                        sm = T.simd.vcadd(T.simd.vadd(e0, e1, mask), mask)
                        sm_all = T.simd.vdupv(sm, mask)
                        # 5. Normalize; only the final result is stored in UB.
                        T.simd.vsts(y[i, 0], T.simd.vdiv(e0, sm_all, mask), mask)
                        T.simd.vsts(y[i, 64], T.simd.vdiv(e1, sm_all, mask), mask)

                T.copy(y, Y[begin : begin + rows, :])

    return tilelang.compile(main, out_idx=-1)
```

每个 tile 的数据路径变为：**GM → a（UB）→ 寄存器内完成五步 → y（UB）→ GM**。中间计算不再往返 UB，但输入搬入和最终输出写回仍然需要。以 rows=8 为例，a、y 单份合计 `2 × 8 × 128 × 4 = 8192 B`，双 buffer 合计 16 KiB；寄存器占用属于另一类资源，不能忽略。列数增大时，需要重新评估寄存器容量，不能无限增加保留在寄存器中的数据。

完整实现与运行入口分别为 [softmax_fused.py](src/softmax_fused.py) 和 [run_softmax_fused.py](src/run_softmax_fused.py)。在本章目录执行：

```bash
source src/env.sh
python src/run_softmax_fused.py
```

该入口沿用本节输入范围，以 `rtol=1e-5, atol=1e-6` 检查最终输出与 CPU Softmax 是否一致、每行和是否接近 1，并检查输出有限且非负。精简版只输出 Y，而前面的教学实现输出五份结果，因此不能直接用此前五输出版本的性能表评价合并 VF 的收益；若进一步比较性能，应先统一输出要求。

**正确性验证**：`(8,128)` 单核单 buffer，以及 `(9216,128)` 的 36 核单、双 buffer，共三组配置；每组检查随机值、全零、常数和大幅值四类输入，12 组均通过。最终输出与 CPU Softmax 的最大绝对误差为 4.470e-08，每行和检查通过。

**五步合一版本的实际运行结果**

下面运行完整验证程序，检查上文的三组配置、每组四类输入。

In [4]:
run_checked("src/run_softmax_fused.py")

{"M": 8, "blocks": 1, "stages": 1, "case": "random", "max_abs_error": 7.450580596923828e-09, "status": "PASS"}
{"M": 8, "blocks": 1, "stages": 1, "case": "zero", "max_abs_error": 0.0, "status": "PASS"}
{"M": 8, "blocks": 1, "stages": 1, "case": "constant", "max_abs_error": 0.0, "status": "PASS"}
{"M": 8, "blocks": 1, "stages": 1, "case": "large", "max_abs_error": 1.6940658945086007e-21, "status": "PASS"}
{"M": 9216, "blocks": 36, "stages": 1, "case": "random", "max_abs_error": 4.470348358154297e-08, "status": "PASS"}
{"M": 9216, "blocks": 36, "stages": 1, "case": "zero", "max_abs_error": 0.0, "status": "PASS"}
{"M": 9216, "blocks": 36, "stages": 1, "case": "constant", "max_abs_error": 0.0, "status": "PASS"}
{"M": 9216, "blocks": 36, "stages": 1, "case": "large", "max_abs_error": 1.6940658945086007e-21, "status": "PASS"}
{"M": 9216, "blocks": 36, "stages": 2, "case": "random", "max_abs_error": 4.470348358154297e-08, "status": "PASS"}
{"M": 9216, "blocks": 36, "stages": 2, "case": "zero"

## 7. 章末验收与回顾

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">检查层次</th>
<th style="text-align: left; vertical-align: top;">应提交的内容</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">单核功能</td>
<td style="text-align: left; vertical-align: top;">五步对应关系、五个完整输出、最终 Softmax 与行和检查</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">tiling 分核</td>
<td style="text-align: left; vertical-align: top;">行 tile 大小、索引推导、整除条件、blocks=1 与 36 的完整结果</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">多 buffer 流水</td>
<td style="text-align: left; vertical-align: top;">六份 buffer 的内存份数与 UB 计算、相同配置的 msprof 数据、性能结论</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">五步合一</td>
<td style="text-align: left; vertical-align: top;">说明寄存器中间结果的数据流，验证仅输出 Y 的精简实现，区分它与五输出教学实现</td>
</tr>
</tbody>
</table>

本章从简单 Vector 算子的读入、计算、写回出发，学会了按 UB 容量分块、按索引分核、利用多 buffer 排布流水，并理解了两种向量计算表达。行 Softmax 将这些知识与归约组合在同一实现中；五步合一进一步说明，中间结果保留在寄存器中可以减少 UB 读写。下一章进入 GEMM，继续使用分块、分核与流水的思路。

完成各档后，对照下面的解析，并核对每一步的完整输出。

<details>
<summary>展开练习解析</summary>

简单档：SimdVF 内全部采用向量寄存器微指令，先用 vmax/vadd 合并多组数据，再用 vcmax/vcadd 得到行归约结果；通过 BRC_B32 广播行标量。`softmax_single` 完整执行 amax、减最大值、exp、对 exp 求和、除以行和；输出 Max、Shift、Exp、Sum、Y，不能只实现原输入的 amax/sum。

中等档：`softmax_kernel` 按完整行组成 tile。`begin=bx*(M//blocks)+it*rows`，所有读入和写回使用同一行偏移；列维仍完整，所以归约不跨核。

困难档：`softmax_pipeline` 对 a、mx、shifted、e、sm、y 都配置 buffer 内存份数，并用 T.Pipelined 遍历相同的行 tile。rows=8、C=128 时每份元素容量为 16.0625 KiB，双 buffer 32.125 KiB，三 buffer 48.1875 KiB。

验收：四类输入逐一核对五个输出，再比较 torch.softmax 与行和；性能使用相同五输出实现的 msprof kernel 耗时。运行对应答案完成设备检查；按正文命令采集当前环境的性能，原表格保留课程参考测量值。

五步合一：以同一行的两个寄存器组完成最大值、平移、指数、求和与归一化；中间结果不再往返UB，只写回Y。精度阈值与前面最终输出一致，为rtol=1e-5、atol=1e-6；性能比较必须先统一输出要求。

</details>

[本章导航](README.md) · [上一节](03.05_execution_modes.ipynb)